# 🧩 TinyRecursiveModels - Local Evaluation & Test-Time Adaptation (TTA)

This notebook is a **restructured, local-friendly version** of the official Kaggle `arc2-trm-v31.ipynb` notebook. It has been stripped of Kaggle paths, environment-specific symlink hacks, and fine-tuned for robust local execution.

### Key Improvements Built-in:
1. **No Symlinks Needed:** Runs directly in your workspace repository root.
2. **Prefix-Robust Checkpoint Loading:** Strips compiled prefixes (`_orig_mod.`) dynamically to prevent runtime mismatches.
3. **Dynamic Puzzle Embedding Resizing:** Automatically matches and mean-initializes puzzle embedding dimensions if the evaluation dataset contains a different number of puzzles than the pre-training checkpoint.
4. **No strict check constraints:** Safely ignores unmatched parameters for seamless plug-and-play checkpoint evaluation.

## 1. Optional: Install Local Dependencies
Run this cell if you need to install required packages inside your local environment.

In [8]:

# ── 1. Environment and Path Setup ───────────────────────────────────────────
import os
import sys
from pathlib import Path

os.chdir("/root/EdgeTRM")
print("Working Directory:", os.getcwd())
# !git fetch origin
# !git reset --hard origin/main
!git pull origin main
# Add TinyRecursiveModels to system path
repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))
print("trm_root added to sys.path:", trm_root)

Working Directory: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 51.00 KiB | 187.00 KiB/s, done.
From https://github.com/Seqaeon/EdgeTRM
 * branch            main       -> FETCH_HEAD
   fbe36aa..eb68cb3  main       -> origin/main
Updating fbe36aa..eb68cb3
Fast-forward
 TinyRecursiveModels/eval-arc-local.py |    43 +-
 arc2-trm-local.ipynb                  | 10799 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 eval-arc-local.py                     |    43 +-
 3 files changed, 10834 insertions(+), 51 deletions(-)
trm_root added to sys.path: /__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/TinyRecursiveModels


In [10]:
import os
os.chdir('/root/EdgeTRM/TinyRecursiveModels')


In [3]:

!uv pip install --system {trm_root}
%uv pip install einops

Using Python 3.12.6 environment at: /usr/local
Resolved 64 packages in 2.69s
Building antlr4-python3-runtime==4.9.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
Building antlr4-python3-runtime==4.9.3
Building tiny-recursive-models @ file:///__modal/volumes/vo-lrr6QFBkMRRvzsKqC3rl4f/T
Building adam-atan2==0.0.3
⠙ Preparing packages... (0/35)
gitdb      ------------------------------     0 B/61.32 KiB
Bui

In [4]:
# ! pip install hydra-core==1.3.2 adam_atan2_pytorch==0.2.4 argdantic==1.3.3 coolname==2.2.0 tqdm pydantic omegaconf

## 2. Generate Augmented Local Dataset
This step runs the dataset builder to compile the augmented inputs, labels, and puzzle indices. Set `--num-aug` (e.g. `128` or `1000` for paper-alignment).

In [20]:
# Create data1 and copy the evaluation challenges and solutions
! mkdir -p data1
! rm -rf data1/arc-agi_test_challenges.json
! rm -rf data1/arc-agi_test_solutions.json
! cp /root/EdgeTRM/arc-prize-2025/arc-agi_evaluation_challenges.json data1/arc-agi_test_challenges.json
! cp /root/EdgeTRM/arc-prize-2025/arc-agi_evaluation_solutions.json data1/arc-agi_test_solutions.json


# Build the augmented dataset
! python -m dataset.build_arc_dataset \
  --input-file-prefix ./data1/arc-agi \
  --output-dir ./data1/arc2test-aug-128 \
  --subsets test \
  --test-set-name test \
  --num-aug 128

[Puzzle da515329] augmentation not full, only 72
Total puzzles: 120
Total puzzle IDs (including <blank>): 15424


## 3. Run Joint Adaptation & Evaluation (TTA)
Run the robust local evaluation pipeline using `torchrun`. You can easily swap your checkpoint path, dataset path, epochs, and other hyperparameters here.

In [28]:
# Standard local evaluation execution. Adjust checkpoint and data paths below.
!HYDRA_FULL_ERROR=1 WANDB_MODE=disabled torchrun --standalone --nnodes=1 --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 \
  eval-arc-local.py \
  arch=trm \
  data_paths="[./data1/arc2test-aug-128]" \
  arch.L_layers=2 \
  arch.H_cycles=4 arch.L_cycles=4 arch.halt_max_steps=10 \
  freeze_weights=False \
  +load_checkpoint=./step_275886 \
  +checkpoint_path=./eval_checkpoint \
  eval_interval=4000 \
  epochs=4000 \
  global_batch_size=128 \
  ema=True \
  lr_warmup_steps=200 \
  lr=0.0001



TinyRecursiveReasoningModel_ACTV1(
  (inner): TinyRecursiveReasoningModel_ACTV1_Inner(
    (embed_tokens): CastedEmbedding()
    (lm_head): CastedLinear()
    (q_head): CastedLinear()
    (puzzle_emb): CastedSparseEmbedding()
    (rotary_emb): RotaryEmbedding()
    (L_level): TinyRecursiveReasoningModel_ACTV1ReasoningModule(
      (layers): ModuleList(
        (0-1): 2 x TinyRecursiveReasoningModel_ACTV1Block(
          (self_attn): Attention(
            (qkv_proj): CastedLinear()
            (o_proj): CastedLinear()
          )
          (mlp): SwiGLU(
            (gate_up_proj): CastedLinear()
            (down_proj): CastedLinear()
          )
        )
      )
    )
  )
)
Loading checkpoint ./step_275886
Resizing puzzle embedding weights dynamically from torch.Size([1041208, 512]) to torch.Size([15424, 512])...
  0%|                                                  | 0/2796 [00:00<?, ?it/s]{'num_params': 6829058}
./eval_checkpoint
Setup EMA
[Rank 0, World Size 1]: Epoch 0
TRAIN
10

## 4. Local Score Validation & Submission Parsing
This cell loads your generated `submission.json` and evaluates the exact match accuracy locally.

In [30]:
import os
import json
import numpy as np

submission_file = "/root/EdgeTRM/TinyRecursiveModels/eval_checkpoint/evaluator_ARC_step_2790/submission.json"
if not os.path.exists(submission_file):
    import glob
    sub_dirs = glob.glob("eval_checkpoint*/evaluator*/submission.json")
    if sub_dirs:
        submission_file = sub_dirs[0]
        print(f"Found submission file dynamically at: {submission_file}")

if os.path.exists(submission_file):
    with open(submission_file, "r") as f:
        submission = json.load(f)
    print(f"Loaded submission file containing {len(submission)} puzzles.")

    # Try matching solution files dynamically
    truth_file = "./data1/arc-agi_test_solutions.json"
    if not os.path.exists(truth_file):
        truth_file = "arc-prize-2025/arc-agi_evaluation_solutions.json"
    if os.path.exists(truth_file):
        with open(truth_file, "r") as f:
            truth = json.load(f)
        
        scores = []
        for puzzle_name, test_attempts in submission.items():
            if puzzle_name in truth:
                puzzle_solution = truth[puzzle_name]
                puzzle_score = 0
                for tid, test_attempt in enumerate(test_attempts):
                    if tid < len(puzzle_solution):
                        sol = puzzle_solution[tid]
                        if test_attempt["attempt_1"] == sol or test_attempt["attempt_2"] == sol:
                            puzzle_score += 1
                scores.append(puzzle_score / len(test_attempts))
        
        if scores:
            print(f"Exact Match Accuracy: {sum(scores) / len(scores) * 100:.2f}%")
        else:
            print("No matching solution keys found in solution file.")
else:
    print("Submission file not found yet. Please run step 3 first!")

Loaded submission file containing 120 puzzles.
Exact Match Accuracy: 9.31%


In [26]:
import json
import os

# 1. Load generated submission keys
submission_file = "./eval_checkpoint/evaluator_ARC_step_1/submission.json"
sub_keys = []
if os.path.exists(submission_file):
    with open(submission_file, "r") as f:
        sub = json.load(f)
    sub_keys = list(sub.keys())
    print(f"🔑 [Submission] Keys ({len(sub_keys)} items): {sub_keys[:5]}")
else:
    print(f"❌ Submission file not found at: {submission_file}")

# 2. Load Evaluation Solutions keys
truth_file = "./data1/arc-agi_test_solutions.json"  # or arc-agi_evaluation_solutions.json
if not os.path.exists(truth_file):
    truth_file = "arc-prize-2025/arc-agi_evaluation_solutions.json"
    
truth_keys = []
if os.path.exists(truth_file):
    with open(truth_file, "r") as f:
        truth = json.load(f)
    truth_keys = list(truth.keys())
    print(f"🔑 [Solutions] Keys ({len(truth_keys)} items): {truth_keys[:5]}")
else:
    print(f"❌ Solutions file not found.")

# 3. Load Evaluation Challenges keys
challenges_file = "./data1/arc-agi_test_challenges.json"
if not os.path.exists(challenges_file):
    challenges_file = "/root/EdgeTRM/arc-prize-2025/arc-agi_evaluation_challenges.json"

chall_keys = []
if os.path.exists(challenges_file):
    with open(challenges_file, "r") as f:
        challenges = json.load(f)
    chall_keys = list(challenges.keys())
    print(f"🔑 [Challenges] Keys ({len(chall_keys)} items): {chall_keys[:5]}")
else:
    print(f"❌ Challenges file not found.")

# 4. Compare Intersections
if sub_keys:
    if truth_keys:
        truth_overlap = set(sub_keys).intersection(set(truth_keys))
        print(f"\n🤝 Overlap [Submission ∩ Solutions]: {len(truth_overlap)} items")
    if chall_keys:
        chall_overlap = set(sub_keys).intersection(set(chall_keys))
        print(f"🤝 Overlap [Submission ∩ Challenges]: {len(chall_overlap)} items")


🔑 [Submission] Keys (240 items): ['0b148d64', '025d127b', '1e0a9b12', '212895b5', '2037f2c7']
🔑 [Solutions] Keys (120 items): ['0934a4d8', '135a2760', '136b0064', '13e47133', '142ca369']
🔑 [Challenges] Keys (120 items): ['0934a4d8', '135a2760', '136b0064', '13e47133', '142ca369']

🤝 Overlap [Submission ∩ Solutions]: 0 items
🤝 Overlap [Submission ∩ Challenges]: 0 items


In [25]:
import json
import os

# Define paths (adjust directories if needed inside Modal)
aug_dir = "/root/EdgeTRM/TinyRecursiveModels/data1/arc2test-aug-128"
if not os.path.exists(aug_dir):
    aug_dir = "data1/arc2test-aug-128"

challenges_file = "/root/EdgeTRM/arc-prize-2025/arc-agi_evaluation_challenges.json"
if not os.path.exists(challenges_file):
    challenges_file = "arc-prize-2025/arc-agi_evaluation_challenges.json"

# 1. Load Compiled Dataset Identifiers
identifiers_file = os.path.join(aug_dir, "identifiers.json")
compiled_ids = []
if os.path.exists(identifiers_file):
    with open(identifiers_file, "r") as f:
        compiled_ids = json.load(f)
    # Strip any padding/blanks
    valid_compiled_ids = [pid for pid in compiled_ids if pid != "<blank>"]
    print(f"📦 [Compiled Dataset] Loaded {len(valid_compiled_ids)} valid puzzle IDs from identifiers.json")
    print(f"👉 Sample compiled IDs: {valid_compiled_ids[:10]}")
else:
    print(f"❌ Could not find identifiers.json at: {identifiers_file}")
    valid_compiled_ids = []

# 2. Load Raw Evaluation Challenges
if os.path.exists(challenges_file):
    with open(challenges_file, "r") as f:
        challenges = json.load(f)
    raw_challenge_keys = list(challenges.keys())
    print(f"\n🧩 [Raw Challenges] Loaded {len(raw_challenge_keys)} puzzle keys from {os.path.basename(challenges_file)}")
    print(f"👉 Sample challenge IDs: {raw_challenge_keys[:10]}")
else:
    print(f"❌ Raw challenges file not found at: {challenges_file}")
    raw_challenge_keys = []

# 3. Verify Alignment
if valid_compiled_ids and raw_challenge_keys:
    matches = set(valid_compiled_ids).intersection(set(raw_challenge_keys))
    mismatches = set(valid_compiled_ids) - set(raw_challenge_keys)
    
    print(f"\n📊 [Alignment Analysis]")
    print(f"✅ Matching puzzle IDs: {len(matches)} / {len(valid_compiled_ids)}")
    if mismatches:
        print(f"⚠️ Compiled IDs NOT in raw challenges ({len(mismatches)} items): {list(mismatches)[:10]}")
    else:
        print("🎉 Perfect alignment! All compiled puzzle IDs exist inside the raw challenges file.")


📦 [Compiled Dataset] Loaded 15423 valid puzzle IDs from identifiers.json
👉 Sample compiled IDs: ['5961cc34', '5961cc34|||t6|||0356281749', '5961cc34|||t6|||0812943567', '5961cc34|||t2|||0724986531', '5961cc34|||t6|||0589462137', '5961cc34|||t6|||0638295417', '5961cc34|||t6|||0189263754', '5961cc34|||t4|||0132874659', '5961cc34|||t2|||0238596147', '5961cc34|||t5|||0682731549']

🧩 [Raw Challenges] Loaded 120 puzzle keys from arc-agi_evaluation_challenges.json
👉 Sample challenge IDs: ['0934a4d8', '135a2760', '136b0064', '13e47133', '142ca369', '16b78196', '16de56c4', '1818057f', '195c6913', '1ae2feb7']

📊 [Alignment Analysis]
✅ Matching puzzle IDs: 120 / 15423
⚠️ Compiled IDs NOT in raw challenges (15303 items): ['e376de54|||t2|||0423718596', 'b10624e5|||t1|||0678539142', 'cb2d8a2c|||t4|||0638912457', '88bcf3b4|||t2|||0827439156', 'abc82100|||t7|||0938274165', '45a5af55|||t0|||0942715638', '89565ca0|||t3|||0751329486', '7ed72f31|||t5|||0791584236', 'b10624e5|||t2|||0651729483', '6e453dd6|